# SpikeInterface pipeline for Tank Lab

In [ ]:
import spikeextractors as se
import spiketoolkit as st
import spikesorters as ss
import spikecomparison as sc
import spikewidgets as sw

from nwb_conversion_tools.conversion_tools import save_si_object
from tank_lab_to_nwb import TowersProcessedNWBConverter

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint
%matplotlib notebook

In [ ]:
stub_test = True
nsec_stub = 10

In [ ]:
spikeinterface_folder = Path('spikeinterface')
spikeinterface_folder.mkdir(parents=True, exist_ok=True)

## 1) Load AP recordings, LF recordings and TTL signals

In [ ]:
base_data_path = Path("/Users/abuccino/Documents/Data/catalyst/brody/A256_bank1_2020_09_30_g0")
# base_data_path = Path("D:/Neuropixels/Neuropixels/A256_bank1_2020_09_30/A256_bank1_2020_09_30_g0")
ap_bin_path = base_data_path / "A256_bank1_2020_09_30_g0_t0.imec0.ap.bin"
lf_bin_path = base_data_path / "A256_bank1_2020_09_30_g0_t0.imec0.lf.bin"

In [ ]:
recording_folder = ap_bin_path.parent

In [ ]:
recording_ap = se.SpikeGLXRecordingExtractor(ap_bin_path)

In [ ]:
recording_lf = se.SpikeGLXRecordingExtractor(lf_bin_path)

In [ ]:
print(f"Sampling frequency AP: {recording_ap.get_sampling_frequency()}")
print(f"Sampling frequency LF: {recording_lf.get_sampling_frequency()}")      

### Load TTL signals

In [ ]:
ttl, states = recording_ap.get_ttl_events()
rising_times = ttl[states==1]

In [ ]:
start_time = recording_ap.frame_to_time(rising_times[0])

In [ ]:
start_frame_ap = int(recording_ap.time_to_frame(start_time))
start_frame_lf = int(recording_lf.time_to_frame(start_time))
print(f"Start frame AP: {start_frame_ap}")
print(f"Start frame LF: {start_frame_lf}")    

### Synchronize recording

In [ ]:
recording_ap_sync = se.SubRecordingExtractor(recording_ap, start_frame=start_frame_ap)
recording_lf_sync = se.SubRecordingExtractor(recording_lf, start_frame=start_frame_lf)

### Inspect signals

In [ ]:
w_ts_ap = sw.plot_timeseries(recording_ap, channel_ids=recording_ap.get_channel_ids()[::10])

In [ ]:
w_ts_lf = sw.plot_timeseries(recording_lf, channel_ids=recording_lf.get_channel_ids()[::10])

## 2) Pre-processing

In [ ]:
apply_cmr = True

In [ ]:
if apply_cmr:
    recording_processed = st.preprocessing.common_reference(recording_ap_sync)
else:
    recording_processed = recording_ap_sync

In [ ]:
if stub_test:
    recording_processed = se.SubRecordingExtractor(recording_processed, 
                                                   end_frame=int(nsec_stub * 
                                                                 recording_processed.get_sampling_frequency()))

In [ ]:
num_frames = recording_processed.get_num_frames()

In [ ]:
spike_rate, spike_amps = st.postprocessing.compute_channel_spiking_activity(recording_processed, n_jobs=1, 
                                                                            verbose=True)

In [ ]:
wmap = sw.plot_activity_map(recording_processed, transpose=True, colorbar=True, log=True)

## 3) Run spike sorters

In [ ]:
sorter_list = ['ironclust', 'herdingspikes']

In [ ]:
# Inspect sorter-specific parameters and defaults
for sorter in sorter_list:
    print(f"\n\n{sorter} params description:")
    pprint(ss.get_params_description(sorter))
    print("Default params:")
    pprint(ss.get_default_params(sorter))    

In [ ]:
# user-specific parameters
sorter_params = dict(
    ironclust=dict(),
    herdingspikes=dict(),
)

In [ ]:
sorting_outputs = ss.run_sorters(
    sorter_list=sorter_list, 
    recording_dict_or_list=dict(rec0=recording_processed),
    working_folder=spikeinterface_folder / "working",
    sorter_params=sorter_params, verbose=True, 
    mode="overwrite",
    run_sorter_kwargs={'raise_error': False}
)

The `sorting_outputs` is a dictionary with ("rec_name", "sorter_name") as keys.

In [ ]:
for result_name, sorting in sorting_outputs.items():
    rec_name, sorter = result_name
    print(f"{sorter} found {len(sorting.get_unit_ids())} units")

## 4) Post-processing: extract waveforms, templates, quality metrics, extracellular features

### Set postprocessing parameters

In [ ]:
# Post-processing params
postprocessing_params = st.postprocessing.get_common_params()
pprint(postprocessing_params)

In [ ]:
# (optional) change parameters
postprocessing_params['max_spikes_per_unit'] = 1000  # with None, all waveforms are extracted

### Set quality metric list

In [ ]:
# Quality metrics
qc_list = st.validation.get_quality_metrics_list()
print(f"Available quality metrics: {qc_list}")

In [ ]:
# (optional) define subset of qc
qc_list = ['snr', 'isi_violation', 'firing_rate']

### Set extracellular features

In [ ]:
# Extracellular features
ec_list = st.postprocessing.get_template_features_list()
print(f"Available EC features: {ec_list}")

In [ ]:
# (optional) define subset of ec
ec_list = ['peak_to_valley', 'halfwidth']

### Postprocess all sorting outputs

In [ ]:
for result_name, sorting in sorting_outputs.items():
    rec_name, sorter = result_name
    print(f"Postprocessing recording {rec_name} sorted with {sorter}")
    tmp_folder = spikeinterface_folder / 'tmp' / sorter
    tmp_folder.mkdir(parents=True)
    
    # set local tmp folder
    sorting.set_tmp_folder(tmp_folder)
    
    # compute waveforms
    waveforms = st.postprocessing.get_unit_waveforms(recording_processed, sorting, **postprocessing_params)
    
    # compute templates
    templates = st.postprocessing.get_unit_templates(recording_processed, sorting, **postprocessing_params)
    
    # comput EC features
    ec = st.postprocessing.compute_unit_template_features(recording_processed, sorting,
                                                          feature_names=ec_list, as_dataframe=True)
    # compute QCs
    qc = st.validation.compute_quality_metrics(sorting, recording=recording_processed, 
                                               metric_names=qc_list, as_dataframe=True)
    
    # export to phy
    phy_folder = spikeinterface_folder / 'phy' / sorter
    phy_folder.mkdir(parents=True, exist_ok=True)
    print("Exporting to phy")
    st.postprocessing.export_to_phy(recording_processed, sorting, phy_folder, verbose=True)

### Run phy and load curated data

In [ ]:
print(phy_folder)

In [ ]:
!phy template-gui spikeinterface/phy/ironclust/params.py

In [ ]:
sorting_manual_curated = se.PhySortingExtractor(phy_folder)

In [ ]:
print(f"Ironclust found {len(sorting.get_unit_ids())} units after manual curation")

## 5) Ensemble spike sorting


In [ ]:
if len(sorting_outputs) > 1:
    # retrieve sortings and sorter names
    sorting_list = []
    sorter_names_comp = []
    for result_name, sorting in sorting_outputs.items():
        rec_name, sorter = result_name
        sorting_list.append(sorting)
        sorter_names_comp.append(sorter)
        
    # run multisorting comparison
    mcmp = sc.compare_multiple_sorters(sorting_list=sorting_list, name_list=sorter_names_comp)
    
    # plot agreement results
    w_agr = sw.plot_multicomp_agreement(mcmp)
    
    # extract ensamble sorting
    sorting_ensamble = mcmp.get_agreement_sorting(minimum_agreement_count=2)
    
    print(f"Ensamble sorting among {sorter_list} found: {len(sorting_ensamble.get_unit_ids())} units")

# 6) Automatic curation

In [ ]:
# define curators and thresholds
isi_violation_threshold = 0.5
snr_threshold = 5
firing_rate_threshold = 0.05

In [ ]:
sorting_auto_curated = []
sorter_names_curation = []
for result_name, sorting in sorting_outputs.items():
    rec_name, sorter = result_name
    sorter_names_curation.append(sorter)
    
    # firing rate threshold
    sorting_curated = st.curation.threshold_firing_rates(sorting, duration_in_frames=num_frames,
                                                         threshold=firing_rate_threshold, 
                                                         threshold_sign='less')
    
    # isi violation threshold
    sorting_curated = st.curation.threshold_isi_violations(sorting, duration_in_frames=num_frames,
                                                           threshold=isi_violation_threshold, 
                                                           threshold_sign='greater')
    
    # isi violation threshold
    sorting_curated = st.curation.threshold_snrs(sorting, recording=recording_processed,
                                                 threshold=snr_threshold, 
                                                 threshold_sign='less')
    sorting_auto_curated.append(sorting_curated)

In [ ]:
for sorting_curated, sorter_name in zip(sorting_auto_curated, sorter_names_curation):
    print(f"{sorter_name} found {len(sorting_curated.get_unit_ids())} units after auto curation")

## 7) Save all outputs in spikeinterface folder

In [ ]:
save_si_object("raw", recording_processed, spikeinterface_folder,
               cache_raw=False, include_properties=True, include_features=False)
save_si_object("sorting_ensemble", sorting_ensemble, spikeinterface_folder,
               cache_raw=False, include_properties=True, include_features=False)
save_si_object("sorter1", sorting_list[0], spikeinterface_folder,
               cache_raw=False, include_properties=True, include_features=False)